In [1]:
import os
from typing import Iterator
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableSequence, RunnableConfig
from langchain_core.runnables import RunnableGenerator
from langchain_core.callbacks import BaseCallbackHandler

load_dotenv()

True

In [15]:
class StepLogger(BaseCallbackHandler):

    def on_chain_start(self, serialized, inputs, **kwargs):
        if serialized is None:
            return
        name = serialized.get("name", "unknown")
        print(f"[LOG] Starting: {name}")

    def on_chain_end(self, outputs, **kwargs):
        print(f"[LOG] Step finished.")

In [16]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
parser = StrOutputParser()

In [17]:
headline_prompt = PromptTemplate(
    template="Write a 2-sentence headline summary about this topic:\n\n{topic}",
    input_variables=["topic"],
)

sentiment_prompt = PromptTemplate(
    template=(
        "Analyze the sentiment of news coverage around this topic.\n"
        "Respond with one word (Positive, Negative, or Neutral) followed by one sentence explaining why.\n\n"
        "Topic: {topic}"
    ),
    input_variables=["topic"],
)

headline_chain = headline_prompt | llm | parser
sentiment_chain = sentiment_prompt | llm | parser

In [ ]:
parallel_step = RunnableParallel( headline=headline_chain, sentiment=sentiment_chain, )

In [ ]:
def merge_results(parallel_output):
    merged = ( "Headline Summary:\n" + parallel_output["headline"] + "\n\nSentiment:\n" + parallel_output["sentiment"] )
    return {"merged": merged}

merge_step = RunnableLambda(merge_results)

In [ ]:
def stream_briefing(input_iter: Iterator) -> Iterator[str]:
    input_dict = next(input_iter)

    briefing_prompt = PromptTemplate(
        template=(
            "Using the research notes below, write a clean 5-sentence news briefing.\n\n"
            "{merged}"
        ),
        input_variables=["merged"],
    )

    full_chain = briefing_prompt | llm | parser
    full_text = full_chain.invoke({"merged": input_dict["merged"]})

    sentences = full_text.split(". ")
    for sentence in sentences:
        sentence = sentence.strip()
        if sentence:
            yield sentence + ".\n"

stream_step = RunnableGenerator(stream_briefing)

In [ ]:
full_chain = RunnableSequence( parallel_step, merge_step, stream_step, )

In [ ]:
config = RunnableConfig( tags=["news-briefing"], callbacks=[StepLogger()], )

In [ ]:
topic = "Climate Change and Global Policy"

print(f"Topic: {topic}")
print("Briefing:\n")

for chunk in full_chain.stream({"topic": topic}, config=config):
    print(chunk, end="", flush=True)

Topic: Climate Change and Global Policy
BRIEFING:

[LOG] Starting: PromptTemplate
[LOG] Step finished.
[LOG] Starting: PromptTemplate
[LOG] Step finished.
[LOG] Step finished.
[LOG] Step finished.
[LOG] Step finished.
[LOG] Step finished.
[LOG] Step finished.
[LOG] Starting: PromptTemplate
[LOG] Step finished.
[LOG] Step finished.
[LOG] Step finished.
Global policymakers are facing mounting pressure to develop and implement effective strategies to combat the escalating impacts of climate change.
The need for comprehensive climate policies, international cooperation, and innovative technologies has become increasingly urgent as the world grapples with rising temperatures and devastating natural disasters.
Despite the alarming reports, there is a growing recognition of the importance of transitioning to a more sustainable future to ensure a livable planet for generations to come.
However, criticism of inadequate government responses to the crisis has dominated the news coverage, contribu